In [ ]:
import sys
import importlib
import os
import gc
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.labeling import load_label_dict
from src.utils.preprocessing import prepare_dataset, normalize_features
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
label_dict = load_label_dict("../data/stream_labels.csv")
X, y = prepare_dataset("../data/SMARTPDM_dataset.csv", label_dict)

print("X shape:", X.shape)  # e.g. (num_windows, 2048, 2)
print("y shape:", y.shape)  # e.g. (num_windows,)

In [ ]:
# Reload the preprocessing module to get the latest version
if 'src.utils.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.utils.preprocessing'])

from src.utils.preprocessing import normalize_features

X_norm, scaler = normalize_features(X)

print("Normalized X shape:", X_norm.shape)
print("Original X range - Current: [{:.3f}, {:.3f}], Vibration: [{:.3f}, {:.3f}]".format(
    X[:,:,0].min(), X[:,:,0].max(), X[:,:,1].min(), X[:,:,1].max()))
print("Normalized X range - Current: [{:.3f}, {:.3f}], Vibration: [{:.3f}, {:.3f}]".format(
    X_norm[:,:,0].min(), X_norm[:,:,0].max(), X_norm[:,:,1].min(), X_norm[:,:,1].max()))

In [ ]:
print("Loading and preprocessing data...")

# Load and preprocess data
label_dict = load_label_dict("../data/stream_labels.csv")
print(f"Loaded {len(label_dict)} labels")

# Use a smaller subset for testing to avoid memory issues
print("Preparing dataset (this may take a while)...")
X_raw, y = prepare_dataset("../data/SMARTPDM_dataset.csv")
print(f"Dataset shape: X={X_raw.shape}, y={y.shape}")

# Check memory usage and use smaller sample if needed
total_size_gb = X_raw.nbytes / (1024**3)
print(f"Dataset size: {total_size_gb:.2f} GB")

if total_size_gb > 2:  # If dataset is larger than 2GB, sample it
    print("Dataset is large, sampling 50% for testing...")
    sample_size = len(X_raw) // 2
    indices = np.random.choice(len(X_raw), sample_size, replace=False)
    X_raw = X_raw[indices]
    y = y[indices]
    print(f"Sampled dataset shape: X={X_raw.shape}, y={y.shape}")

print("Normalizing features...")
X_norm, scaler = normalize_features(X_raw)

# Clear raw data to free memory
del X_raw
gc.collect()

print("Splitting dataset...")
# Split into train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train set: X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}, Val: {np.bincount(y_val)}, Test: {np.bincount(y_test)}")

# Clean up
del X_norm, X_temp, y_temp
gc.collect()

print("Data preprocessing completed successfully!")